In [10]:
import numpy as np
from scipy import stats

from Registration.uti import load_registration_results


selected_organs = [
    "brain",
    "skull",
    "heart",
    "aorta",
    "trachea",
    "esophagus",
    "liver",
    "spleen",
    "pancreas",
    "stomach",
    "kidney_left",
    "kidney_right",
    "colon",
    "urinary_bladder",
    "prostate",
    "spinal_cord",
    "vertebrae_L3",
]

s = 4500


a = fr"C:\Users\Sam\Downloads\pet_reg_results\ctsmoothness_l{s}_k10_mar3000_gam1.0.txt"
b = fr"C:\Users\Sam\Downloads\pet_reg_results\baseline_l{s}_k10.txt"


masks_names, dice_before_lists, \
    dice_after_lists_0, tre_before_lists, tre_after_lists_0 = \
        load_registration_results(a)

masks_names, dice_before_lists, \
    dice_after_lists_1, tre_before_lists, tre_after_lists_1 = \
        load_registration_results(b)

selected_indices = [masks_names.index(organ) for organ in selected_organs]




import numpy as np
import ast
from scipy import stats

def _to_float_array(x):
    # If it's a string like "[0.1, 0.2, ...]" parse it
    if isinstance(x, str):
        x = ast.literal_eval(x)

    arr = np.asarray(x, dtype=float).ravel()

    # Remove NaN/inf if any
    arr = arr[np.isfinite(arr)]
    return arr

def print_p(listA, listB, organ_name):
    model_A = _to_float_array(listA)
    model_B = _to_float_array(listB)

    if model_A.shape != model_B.shape:
        raise ValueError(f"{organ_name}: shape mismatch A{model_A.shape} vs B{model_B.shape}")

    # Paired t-test
    t_stat, p_value = stats.ttest_rel(model_A, model_B)

    # Wilcoxon (more robust for non-normal metrics)
    try:
        w_stat, p_w = stats.wilcoxon(model_A, model_B, zero_method="wilcox")
    except ValueError:
        # can happen if all differences are zero
        w_stat, p_w = np.nan, 1.0

    diff = model_A - model_B
    mean_diff = diff.mean()
    std_diff = diff.std(ddof=1) if diff.size > 1 else np.nan
    cohen_d = mean_diff / std_diff if (std_diff not in [0, np.nan] and std_diff != 0) else np.nan

    print(f"\nOrgan: {organ_name}")
    print(f"N = {len(model_A)}")
    print(f"Mean(A) = {model_A.mean():.4f} ± {model_A.std(ddof=1):.4f}")
    print(f"Mean(B) = {model_B.mean():.4f} ± {model_B.std(ddof=1):.4f}")
    print(f"Mean(A-B) = {mean_diff:.4f}")

    print("Paired t-test:    t = {:.4f}, p = {:.4e}".format(t_stat, p_value))
    print("Wilcoxon test:    W = {}, p = {:.4e}".format(w_stat, p_w))
    print("Effect size (paired Cohen's d) = {:.4f}".format(cohen_d))


for organ in selected_organs:
    index = masks_names.index(organ)
    # usage
    print_p(dice_after_lists_0[index], dice_after_lists_1[index], organ)

print("********************************")
for organ in selected_organs:
    index = masks_names.index(organ)
    # usage
    print_p(tre_after_lists_0[index], tre_after_lists_1[index], organ)



Organ: brain
N = 80
Mean(A) = 0.8243 ± 0.0889
Mean(B) = 0.8737 ± 0.0755
Mean(A-B) = -0.0494
Paired t-test:    t = -8.9888, p = 1.0231e-13
Wilcoxon test:    W = 186.0, p = 6.0752e-12
Effect size (paired Cohen's d) = -1.0050

Organ: skull
N = 80
Mean(A) = 0.4866 ± 0.1383
Mean(B) = 0.5823 ± 0.1369
Mean(A-B) = -0.0957
Paired t-test:    t = -12.8004, p = 5.8898e-21
Wilcoxon test:    W = 78.0, p = 1.4048e-13
Effect size (paired Cohen's d) = -1.4311

Organ: heart
N = 80
Mean(A) = 0.7219 ± 0.0987
Mean(B) = 0.7888 ± 0.0795
Mean(A-B) = -0.0669
Paired t-test:    t = -9.2667, p = 2.9342e-14
Wilcoxon test:    W = 198.0, p = 9.0838e-12
Effect size (paired Cohen's d) = -1.0361

Organ: aorta
N = 80
Mean(A) = 0.5466 ± 0.2230
Mean(B) = 0.7354 ± 0.1325
Mean(A-B) = -0.1888
Paired t-test:    t = -12.1463, p = 9.3450e-20
Wilcoxon test:    W = 66.0, p = 9.0944e-14
Effect size (paired Cohen's d) = -1.3580

Organ: trachea
N = 80
Mean(A) = 0.3671 ± 0.2237
Mean(B) = 0.5938 ± 0.1764
Mean(A-B) = -0.2267
Paired t-